In [3]:
import pandas as pd

movies_ml = pd.read_csv(
    "data/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movieId", "title", "genres"]
)

ratings_ml = pd.read_csv(
    "data/ratings.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["userId", "movieId", "rating", "timestamp"]
)

In [1]:
import os

print(os.getcwd())
print(os.listdir())

c:\Users\sejal\PycharmProjects\ML-proj
['.git', '.gitattributes', '.gitignore', '.idea', '.ipynb_checkpoints', '.venv', 'app.py', 'collaborative-filtering.ipynb', 'data', 'Git LFS', 'movie-recommender.ipynb', 'movie_dict.pkl', 'Procfile', 'project-notes.txt', 'README.md', 'requirements.txt', 'screenshot1.png', 'screenshot2.png', 'setup.sh', 'similarity.pkl']


In [2]:
import os

print(os.path.exists("data/movies.dat"))
print(os.path.exists("data/ratings.dat"))


True
True


In [5]:
print(movies_ml.shape)
ratings_ml.shape

(3883, 3)


(1000209, 4)

In [6]:
movies_ml.head()

,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
ratings_ml["rating"].value_counts().sort_index()
#count how many time each rating occurs sorted by rating like 1 , 2 

rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64

In [ ]:
user_movie_matrix = ratings_ml.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)
#Pivot table rearranges the raw data into a matrix by choosing what represents the rows, columns, and values.
#stores which user gave which movie what rating 

user_movie_matrix.shape

(6040, 3706)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

movie_similarity = cosine_similarity(
    user_movie_matrix.T.fillna(0)
)
# with each movie pair by transposing and removing na values we found similarity b/w 
# each movie pair 

movie_similarity.shape

(3706, 3706)

In [ ]:
movie_id_to_index = {
    movie_id: index
    for index, movie_id in enumerate(user_movie_matrix.columns)
}
#Now created dict to map each movie id to its col index 

In [13]:
list(movie_id_to_index.items())[:10]

[(1, 0),
 (2, 1),
 (3, 2),
 (4, 3),
 (5, 4),
 (6, 5),
 (7, 6),
 (8, 7),
 (9, 8),
 (10, 9)]

In [14]:
def collaborative_recommend(movie_id, n=5):
    idx = movie_id_to_index[movie_id]
    
    scores = movie_similarity[idx]
    
    similar_indices = scores.argsort()[::-1][1:n+1]
    
    recommended_ids = [
        user_movie_matrix.columns[i]
        for i in similar_indices
    ]
    
    return recommended_ids

In [15]:
collaborative_recommend(1)

[np.int64(3114), np.int64(1265), np.int64(588), np.int64(2355), np.int64(1270)]

In [ ]:
movie_id_to_title = dict(
    zip(movies_ml["movieId"], movies_ml["title"])
)
#zip makes pairs of corresponding movie id and title 
# and then convert into dict 

In [17]:
[movie_id_to_title[movie_id] for movie_id in collaborative_recommend(1)]

['Toy Story 2 (1999)',
 'Groundhog Day (1993)',
 'Aladdin (1992)',
 "Bug's Life, A (1998)",
 'Back to the Future (1985)']

In [19]:
tmdb_movies = pd.read_csv("tmdb_movie_mapping.csv")

In [20]:
tmdb_movies.head()

,movie_id,title
0,19995,Avatar
1,285,Pirates of the Caribbean: At World's End
2,206647,Spectre
3,49026,The Dark Knight Rises
4,49529,John Carter


In [21]:
import re

def normalize_title(title):
    return re.sub(r'\s*\(\d{4}\)$', '', title).strip().lower()

movies_ml["normalized_title"] = movies_ml["title"].apply(normalize_title)
tmdb_movies["normalized_title"] = tmdb_movies["title"].apply(normalize_title)

In [22]:
movies_ml[["title", "normalized_title"]].head()

,title,normalized_title
0,Toy Story (1995),toy story
1,Jumanji (1995),jumanji
2,Grumpier Old Men (1995),grumpier old men
3,Waiting to Exhale (1995),waiting to exhale
4,Father of the Bride Part II (1995),father of the bride part ii


In [ ]:
common_movies = movies_ml.merge(
    tmdb_movies,
    on="normalized_title", #get only movies whose normalized title match 
    suffixes=("_ml", "_tmdb") # after merge you have 2 cols name titile so add these suffixes to distinguish 
)

In [24]:
common_movies.shape

(894, 6)

In [25]:
common_movies[["title_ml", "title_tmdb"]].head(10)


,title_ml,title_tmdb
0,Toy Story (1995),Toy Story
1,GoldenEye (1995),GoldenEye
2,Nixon (1995),Nixon
3,Cutthroat Island (1995),Cutthroat Island
4,Casino (1995),Casino
5,Sense and Sensibility (1995),Sense and Sensibility
6,Four Rooms (1995),Four Rooms
7,Ace Ventura: When Nature Calls (1995),Ace Ventura: When Nature Calls
8,Money Train (1995),Money Train
9,Get Shorty (1995),Get Shorty


In [ ]:
movie_mapping = common_movies[
    ["movieId", "movie_id", "title_ml"]
].copy()
#"movieId" → MovieLens movie ID
#"movie_id" → TMDB movie ID
#"title_ml" → MovieLens movie title

movie_mapping.head()

,movieId,movie_id,title_ml
0,1,862,Toy Story (1995)
1,10,710,GoldenEye (1995)
2,14,10858,Nixon (1995)
3,15,1408,Cutthroat Island (1995)
4,16,524,Casino (1995)


In [ ]:
matched_movie_ids = set(movie_mapping["movieId"])
#set for faster checking contains movilens movieids 
ratings_matched = ratings_ml[
    ratings_ml["movieId"].isin(matched_movie_ids)
].copy()

#got the ratings only for matched movieId

In [ ]:
ratings_matched.shape #how man

(449144, 4)

In [ ]:
matched_matrix = ratings_matched.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print(matched_matrix.shape)
#earlier we had 894 movies but 8 of them didnt have ratings in Movielens so left with 886 movies 

(6040, 886)


In [30]:
movie_similarity_matched = cosine_similarity(
    matched_matrix.T.fillna(0)
)

movie_similarity_matched.shape

(886, 886)

In [ ]:
matched_movie_id_to_index = {
    movie_id: index
    for index, movie_id in enumerate(matched_matrix.columns)
}
#enumerate gives index,movieid pairs 
#dictionary comprehension reverses into movieid,index

In [32]:
list(matched_movie_id_to_index.items())[:10]

[(1, 0),
 (10, 1),
 (14, 2),
 (15, 3),
 (16, 4),
 (17, 5),
 (18, 6),
 (19, 7),
 (20, 8),
 (21, 9)]

In [33]:
def collaborative_recommend(movie_id, n=5):
    idx = matched_movie_id_to_index[movie_id]

    scores = movie_similarity_matched[idx]

    similar_indices = scores.argsort()[::-1][1:n+1]

    recommended_ids = [
        matched_matrix.columns[i]
        for i in similar_indices
    ]

    return recommended_ids

In [ ]:
collaborative_recommend(1)

In [34]:
[movie_id_to_title[movie_id] for movie_id in collaborative_recommend(1)]

['Toy Story 2 (1999)',
 'Groundhog Day (1993)',
 'Aladdin (1992)',
 'Back to the Future (1985)',
 'Babe (1995)']

In [ ]:
ml_to_tmdb = dict(
    zip(movie_mapping["movieId"], movie_mapping["movie_id"])
)
#zip pairs of movieId(movilens) and movie_id(tmdb)
#and forms dict of these pairs 
#so got movilens id to tmdb id mapping 

In [36]:
ml_to_tmdb[1]

862

In [37]:
collab_ids = collaborative_recommend(1)

tmdb_ids = [
    ml_to_tmdb[movie_id]
    for movie_id in collab_ids
    if movie_id in ml_to_tmdb
]

tmdb_ids

[863, 137, 812, 105, 9598]

In [38]:
import pickle

with open("movie_dict.pkl", "rb") as f:
    movie_dict = pickle.load(f)

with open("similarity.pkl", "rb") as f:
    content_similarity = pickle.load(f)

In [39]:
print(type(movie_dict))
print(type(content_similarity))
print(len(movie_dict))
print(content_similarity.shape)

<class 'dict'>
<class 'numpy.ndarray'>
3
(4806, 4806)


In [40]:
movie_dict.keys()

dict_keys(['movie_id', 'title', 'tags'])

In [49]:
tmdb_ids_ordered = list(movie_dict["movie_id"].values())

In [ ]:
tmdb_id_to_content_index = {
    movie_id: index
    for index, movie_id in enumerate(tmdb_ids_ordered)
}
#created movie id to similarity matrix index mapping 

In [51]:
idx = tmdb_id_to_content_index[862]

print(idx)
print(tmdb_ids_ordered[idx])
print(movie_dict["title"][idx])


1543
862
Toy Story


In [ ]:
idx = tmdb_id_to_content_index[862]

content_scores = content_similarity[idx]

content_indices = content_scores.argsort()[::-1][1:6]

content_tmdb_ids = [
    tmdb_ids_ordered[i]
    for i in content_indices
]

content_tmdb_ids #stores top 5 movies of similar content based on similarity score we calculated 

[863, 10193, 6957, 73247, 34549]

In [ ]:
ml_movie_id = 1  # Toy Story

collab_idx = matched_movie_id_to_index[ml_movie_id]

collab_scores = movie_similarity_matched[collab_idx]

collab_indices = collab_scores.argsort()[::-1][1:]

collab_results = [
    (matched_matrix.columns[i], collab_scores[i])
    for i in collab_indices
    if matched_matrix.columns[i] in ml_to_tmdb
]

collab_results[:5] # stores matched ids, score 

[(np.int64(3114), np.float64(0.6331037415587969)),
 (np.int64(1265), np.float64(0.6108261558354962)),
 (np.int64(588), np.float64(0.6058491100871751)),
 (np.int64(1270), np.float64(0.5701254186194334)),
 (np.int64(34), np.float64(0.5636370913965632))]

In [ ]:
collab_tmdb_results = [
    (ml_to_tmdb[movie_id], score)
    for movie_id, score in collab_results
]

collab_tmdb_results[:5] # stores movieid ,score content based 

[(863, np.float64(0.6331037415587969)),
 (137, np.float64(0.6108261558354962)),
 (812, np.float64(0.6058491100871751)),
 (105, np.float64(0.5701254186194334)),
 (9598, np.float64(0.5636370913965632))]

In [ ]:
content_top = content_scores.argsort()[::-1][1:51]
#got top 50 content similarity 
content_candidates = {
    tmdb_ids_ordered[i]: content_scores[i]
    for i in content_top
}
#store movie id , content score 


In [ ]:
collab_top = collab_results[:50]

collab_candidates = {
    ml_to_tmdb[movie_id]: score
    for movie_id, score in collab_top
}
#here also contains top 50 collab score movie ids 

In [58]:
candidate_ids = set(content_candidates) | set(collab_candidates)

len(candidate_ids)

99

In [ ]:
from sklearn.preprocessing import MinMaxScaler

all_scores = []

for movie_id in candidate_ids:
    all_scores.append([
        content_candidates.get(movie_id, 0),
        collab_candidates.get(movie_id, 0)
    ])

#for all 99 movies you are creating content_Score, collab_Score pairs and if one of value doesnt 
#exist give it 0 
#now on those pair minmaxscaler is applied to perform normalization 

scaler = MinMaxScaler()
normalized_scores = scaler.fit_transform(all_scores)

In [ ]:

content_rank = {
    movie_id: rank
    for rank, movie_id in enumerate(
        sorted(content_candidates, key=content_candidates.get, reverse=True)
    ) #sort content_Cand based on score and assign rank to them 
    #highest score rank 1 
}

collab_rank = {
    movie_id: rank
    for rank, movie_id in enumerate(
        sorted(collab_candidates, key=collab_candidates.get, reverse=True)
    )
}

hybrid_scores = {}

for movie_id in candidate_ids:
    c_rank = content_rank.get(movie_id, 100)
    cf_rank = collab_rank.get(movie_id, 100)
    #if rank isnt assign to it . make it 100 
    hybrid_scores[movie_id] = (
        0.5 * (1 / (c_rank + 1)) +
        0.5 * (1 / (cf_rank + 1))
    )

In [63]:
top_hybrid = sorted(
    hybrid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:5]

top_hybrid

[(863, 1.0),
 (137, 0.25495049504950495),
 (10193, 0.25495049504950495),
 (6957, 0.1716171617161716),
 (812, 0.1716171617161716)]

In [64]:
tmdb_to_ml = {
    tmdb_id: ml_id
    for ml_id, tmdb_id in ml_to_tmdb.items()
}

In [65]:
tmdb_to_ml[862]

1

In [66]:
def hybrid_recommend(movie_id, n=5):
    # Content scores
    content_idx = tmdb_id_to_content_index[movie_id]
    content_scores = content_similarity[content_idx]

    content_indices = content_scores.argsort()[::-1][1:51]

    content_candidates = {
        tmdb_ids_ordered[i]: content_scores[i]
        for i in content_indices
    }

    # Collaborative scores
    ml_id = tmdb_to_ml.get(movie_id)

    if ml_id is None:
        return list(content_candidates.keys())[:n]

    collab_idx = matched_movie_id_to_index.get(ml_id)

    if collab_idx is None:
        return list(content_candidates.keys())[:n]

    collab_scores = movie_similarity_matched[collab_idx]
    collab_indices = collab_scores.argsort()[::-1][1:51]

    collab_candidates = {
        ml_to_tmdb[matched_matrix.columns[i]]: collab_scores[i]
        for i in collab_indices
        if matched_matrix.columns[i] in ml_to_tmdb
    }

    # Candidate pool
    candidate_ids = set(content_candidates) | set(collab_candidates)

    # Rank-based hybrid scoring
    content_rank = {
        movie_id: rank
        for rank, movie_id in enumerate(
            sorted(content_candidates,
                   key=content_candidates.get,
                   reverse=True)
        )
    }

    collab_rank = {
        movie_id: rank
        for rank, movie_id in enumerate(
            sorted(collab_candidates,
                   key=collab_candidates.get,
                   reverse=True)
        )
    }

    hybrid_scores = {}

    for candidate in candidate_ids:
        c_rank = content_rank.get(candidate, 100)
        cf_rank = collab_rank.get(candidate, 100)

        hybrid_scores[candidate] = (
            0.5 / (c_rank + 1) +
            0.5 / (cf_rank + 1)
        )

    return [
        movie_id
        for movie_id, score in sorted(
            hybrid_scores.items(),
            key=lambda x: x[1],
            reverse=True
        )[:n]
    ]

In [67]:
hybrid_recommend(862)

[863, 137, 10193, 6957, 812]

In [68]:
id_to_title = dict(
    zip(movie_dict["movie_id"].values(),
        movie_dict["title"].values())
)

In [69]:
[id_to_title[movie_id] for movie_id in hybrid_recommend(862)]

['Toy Story 2',
 'Groundhog Day',
 'Toy Story 3',
 'The 40 Year Old Virgin',
 'Aladdin']

In [70]:
import pickle

with open("collaborative_model.pkl", "wb") as f:
    pickle.dump({
        "movie_similarity": movie_similarity_matched,
        "movie_id_to_index": matched_movie_id_to_index,
        "matched_matrix_columns": list(matched_matrix.columns),
        "ml_to_tmdb": ml_to_tmdb,
        "tmdb_to_ml": tmdb_to_ml
    }, f)

In [71]:
from sklearn.model_selection import train_test_split

train_ratings, test_ratings = train_test_split(
    ratings_ml,
    test_size=0.2,
    random_state=42
)

print(train_ratings.shape)
print(test_ratings.shape)

(800167, 4)
(200042, 4)


In [72]:
test_liked = test_ratings[
    test_ratings["rating"] >= 4
]

In [73]:
test_liked.shape


(115162, 4)

In [74]:
test_liked.head()

,userId,movieId,rating,timestamp
899739,5440,904,5,959995181
55687,368,3717,4,976311423
63727,425,1721,4,976283587
781894,4668,2011,4,963801714
472805,2907,173,5,971820825


In [75]:
train_matrix = train_ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

print(train_matrix.shape)

(6040, 3683)


In [76]:
train_similarity = cosine_similarity(
    train_matrix.T.fillna(0)
)

print(train_similarity.shape)

(3683, 3683)


In [77]:
def train_recommend(movie_id, n=5):
    if movie_id not in train_matrix.columns:
        return []

    idx = train_matrix.columns.get_loc(movie_id)

    scores = train_similarity[idx]

    indices = scores.argsort()[::-1][1:n+1]

    return [
        train_matrix.columns[i]
        for i in indices
    ]

In [78]:
train_recommend(1)

[np.int64(3114), np.int64(1265), np.int64(1270), np.int64(588), np.int64(2355)]

In [79]:
liked_movies_by_user = (
    test_liked
    .groupby("userId")["movieId"]
    .apply(set)
    .to_dict()
)

In [80]:
liked_movies_by_user.get(1, set())

{608, 1035, 1193, 1246, 1836, 2018, 2398, 2692, 2804, 3105, 3186}

In [81]:
precisions = []

for user_id, liked_movies in liked_movies_by_user.items():

    # Movies this user rated in training
    user_train_movies = set(
        train_ratings[
            train_ratings["userId"] == user_id
        ]["movieId"]
    )

    # Pick a movie the user rated in training
    if not user_train_movies:
        continue

    movie_id = next(iter(user_train_movies))

    # Generate 5 recommendations
    recommendations = train_recommend(
        movie_id,
        n=5
    )

    # How many recommended movies were liked in test?
    hits = len(
        set(recommendations) & liked_movies
    )

    precision = hits / 5

    precisions.append(precision)

In [82]:
precision_at_5 = sum(precisions) / len(precisions)

print("Precision@5:", precision_at_5)

Precision@5: 0.05095095095095096
